In [ ]:
import pandas as pd
import numpy as np
from sklearn.multioutput import MultiOutputRegressor
from sklearn.linear_model import Ridge, LinearRegression
from sklearn.ensemble import ExtraTreesRegressor
from sklearn.ensemble import RandomForestRegressor
import xgboost as xgb
pd.set_option('display.max_columns', 200)
pd.set_option('display.max_colwidth', 20)
from sklearn.model_selection import RandomizedSearchCV
import numpy as np
from scipy.stats import randint, uniform
import joblib
from sklearn.model_selection import cross_val_predict
import csv

### Data Preprocessing

In [424]:
df = pd.read_csv("./datasets/new_raw.csv")
first = ["name", "nvar"]
df = df[first + [c for c in df.columns if c not in first]]
df["nvar"].value_counts
df = df.sort_values(["name", "nvar", "mem"]).reset_index()
df

,index,name,nvar,nvmops,objective,eval_duration_solver,extract_duration_solver,stats_elapsed_time,dual_feas,status,timed_bytes,timed_time,timed_gctime,nlp_warmup_time,mem,neval_grad,iter,source_solver,problem,neval_obj,timestamp_solver,error_solver,vector_type,extract_duration_problem,highest_degree (ExprTree),nln_nnzj,adbackend_hessian_backend_type,minimize,count_plus_minus,alloc_obj,jtprod_available,has_equalities,time_hprod,adbackend_jprod_residual_backend_type,adbackend_ghjvprod_backend_type,nnln,jprod_residual_available,nlvb,nlvo,nlp_type,unconstrained,adbackend_jacobian_backend_type,adbackend_jprod_backend_type,hprod_residual_available,jac_available,hess_residual_available,adbackend_gradient_backend_type,float_type,ncon,is_nls,nnzo,nlvc,tree_length,count_trigonometric_function,matrix_free,nlin,time_obj,tree_depth,is_constant (ExprTree),nnzj,alloc_hprod,adbackend_jacobian_residual_backend_type,generator,alloc_hess,hess_available,count_exponential_function,islp,count_op_function,error_problem,adbackend_hprod_residual_backend_type,equality_constrained,time_grad,source_problem,has_bounds,grad_available,clinvals_nnz,jac_residual_available,bound_constrained,jprod_available,eval_duration_problem,adbackend_hessian_residual_backend_type,nequ,adbackend_jtprod_residual_backend_type,has_inequalities,is_linear (ExprTree),linearly_constrained,adbackend_jtprod_backend_type,hprod_available,jtprod_residual_available,alloc_grad,inequality_constrained,time_hess,lin_nnzj,nnzh,timestamp_problem,adbackend_hprod_backend_type,error_type,total_alloc
0,15939,arglina,100,1.0,5.000000e+01,0.319062,0.011306,0.000135,7.841650e-15,first_order,18576.0,0.000164,0.000000,0.377710,1,4.0,1.0,OptimizationProb...,OptimizationProb...,4.0,2026-02-28 18:33...,NaN,Vector{Float64},4.309682,2.0,0.0,ADNLPModels.Empt...,1.0,305.0,34515160.0,0.0,0.0,NaN,ADNLPModels.Empt...,ADNLPModels.Empt...,0.0,NaN,100.0,100.0,ADNLPModel{Float...,1.0,ADNLPModels.Empt...,ADNLPModels.Empt...,NaN,0.0,NaN,ADNLPModels.Forw...,Float64,0.0,0.0,100.0,100.0,10404.0,0.0,1.0,0.0,0.376715,9.0,False,0.0,NaN,ADNLPModels.Empt...,OptimizationProb...,NaN,1.0,0.0,0.0,509.0,NaN,ADNLPModels.Empt...,0.0,0.464686,OptimizationProb...,0.0,1.0,0.0,NaN,0.0,0.0,1.980544,ADNLPModels.Empt...,NaN,ADNLPModels.Empt...,0.0,False,0.0,ADNLPModels.Empt...,1.0,NaN,4.261110e+07,0.0,NaN,0.0,5.050000e+03,2026-01-31 17:27...,ADNLPModels.Forw...,NaN,7.712626e+07
1,15940,arglina,100,1.0,5.000000e+01,0.319062,0.011309,0.000151,7.841650e-15,first_order,22120.0,0.000199,0.000000,0.377710,2,2.0,1.0,OptimizationProb...,OptimizationProb...,2.0,2026-02-28 18:33...,NaN,Vector{Float64},4.309682,2.0,0.0,ADNLPModels.Empt...,1.0,305.0,34515160.0,0.0,0.0,NaN,ADNLPModels.Empt...,ADNLPModels.Empt...,0.0,NaN,100.0,100.0,ADNLPModel{Float...,1.0,ADNLPModels.Empt...,ADNLPModels.Empt...,NaN,0.0,NaN,ADNLPModels.Forw...,Float64,0.0,0.0,100.0,100.0,10404.0,0.0,1.0,0.0,0.376715,9.0,False,0.0,NaN,ADNLPModels.Empt...,OptimizationProb...,NaN,1.0,0.0,0.0,509.0,NaN,ADNLPModels.Empt...,0.0,0.464686,OptimizationProb...,0.0,1.0,0.0,NaN,0.0,0.0,1.980544,ADNLPModels.Empt...,NaN,ADNLPModels.Empt...,0.0,False,0.0,ADNLPModels.Empt...,1.0,NaN,4.261110e+07,0.0,NaN,0.0,5.050000e+03,2026-01-31 17:27...,ADNLPModels.Forw...,NaN,7.712626e+07
2,15941,arglina,100,1.0,5.000000e+01,0.319062,0.011275,0.000140,7.841650e-15,first_order,25592.0,0.000182,0.000000,0.377710,3,2.0,1.0,OptimizationProb...,OptimizationProb...,2.0,2026-02-28 18:33...,NaN,Vector{Float64},4.309682,2.0,0.0,ADNLPModels.Empt...,1.0,305.0,34515160.0,0.0,0.0,NaN,ADNLPModels.Empt...,ADNLPModels.Empt...,0.0,NaN,100.0,100.0,ADNLPModel{Float...,1.0,ADNLPModels.Empt...,ADNLPModels.Empt...,NaN,0.0,NaN,ADNLPModels.Forw...,Float64,0.0,0.0,100.0,100.0,10404.0,0.0,1.0,0.0,0.376715,9.0,False,0.0,NaN,ADNLPModels.Empt...,OptimizationProb...,NaN,1.0,0.0,0.0,509.0,NaN,ADNLPModels.Empt...,0.0,0.464686,OptimizationProb...,0.0,1.0,0.0,NaN,0.0,0.0,1.980544,ADNLPModels.Empt...,NaN,ADNLPModels.Empt...,0.0,False,0.0,ADNLPModel

In [425]:
feature_cols_reg = [
                    # Core Features
                    "nvar", 
                    "mem",
                    "tree_length", 
                    "tree_depth", 
                    "time_obj",  # initial eval
                    "time_grad",
                    "extract_duration_problem",
                    
                    # Expression Tree Features
                    "highest_degree (ExprTree)",
                    "count_plus_minus",
                    "count_trigonometric_function",
                    "count_exponential_function",
                    "count_op_function",

                    # allocation
                    "alloc_obj",
                    "alloc_grad",
                    "total_alloc",
                  ]

target_cols_reg = [ "neval_obj",
                    "neval_grad",
                    "timed_bytes"] #(neval_obj, neval_grad, timed_bytes)

target_col_model = ["stats_elapsed_time"]

group_key = ["name", "nvar"]                                                                                                                   # [...]     
all_cols = list(set(feature_cols_reg + target_cols_reg + target_col_model + group_key))
solver_metrics = ['neval_grad', 'neval_obj', 'timed_bytes']
df[solver_metrics] = df[solver_metrics].fillna(0)
df = df[all_cols]
df.isna().sum().sort_values(ascending=False)

nvar                            0
extract_duration_problem        0
tree_depth                      0
tree_length                     0
stats_elapsed_time              0
highest_degree (ExprTree)       0
neval_grad                      0
timed_bytes                     0
time_obj                        0
count_op_function               0
name                            0
alloc_grad                      0
neval_obj                       0
mem                             0
count_plus_minus                0
count_trigonometric_function    0
time_grad                       0
total_alloc                     0
alloc_obj                       0
count_exponential_function      0
dtype: int64

In [426]:
df = df.copy()
df["mem_count"] = df.groupby(["name", "nvar"]).transform('size')
df = df[df["mem_count"] == 100].copy().reset_index(drop=True)
df

,nvar,extract_duration_problem,alloc_obj,total_alloc,time_grad,count_trigonometric_function,count_plus_minus,mem,neval_obj,alloc_grad,name,count_op_function,time_obj,timed_bytes,neval_grad,highest_degree (ExprTree),stats_elapsed_time,tree_length,tree_depth,count_exponential_function,mem_count
0,100,4.309682,34515160.0,7.712626e+07,0.464686,0.0,305.0,1,4.0,4.261110e+07,arglina,509.0,0.376715,18576.0,4.0,2.0,0.000135,10404.0,9.0,0.0,100
1,100,4.309682,34515160.0,7.712626e+07,0.464686,0.0,305.0,2,2.0,4.261110e+07,arglina,509.0,0.376715,22120.0,2.0,2.0,0.000151,10404.0,9.0,0.0,100
2,100,4.309682,34515160.0,7.712626e+07,0.464686,0.0,305.0,3,2.0,4.261110e+07,arglina,509.0,0.376715,25592.0,2.0,2.0,0.000140,10404.0,9.0,0.0,100
3,100,4.309682,34515160.0,7.712626e+07,0.464686,0.0,305.0,4,2.0,4.261110e+07,arglina,509.0,0.376715,29128.0,2.0,2.0,0.000151,10404.0,9.0,0.0,100
4,100,4.309682,34515160.0,7.712626e+07,0.464686,0.0,305.0,5,2.0,4.261110e+07,arglina,509.0,0.376715,32600.0,2.0,2.0,0.000139,10404.0,9.0,0.0,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21495,100000,2813.283680,65309368.0,2.579726e+10,35.399378,0.0,275001.0,96,68.0,2.573195e+10,woods,575001.0,0.073713,314448744.0,59.0,2.0,1279.820757,425000.0,7.0,0.0,100
21496,100000,2813.283680,65309368.0,2.579726e+10,35.399378,0.0,275001.0,97,68.0,2.573195e+10,woods,575001.0,0.073713,317646328.0,59.0,2.0,1112.816634,425000.0,7.0,0.0,100
21497,100000,2813.283680,65309368.0,2.579726e+10,35.399378,0.0,275001.0,98,68.0,2.573195e+10,woods,575001.0,0.073713,320848008.0,59.0,2.0,1162.156092,425000.0,7.0,0.0,100
21498,100000,2813.283680,65309368.0,2.579726e+10,35.399378,0.0,275001.0,99,68.0,2.573195e+10,woods,575001.0,0.073713,324045592.0,59.0,2.0,1396.521273,425000.0,7.0,0.0,100


In [427]:
# unique instances: one row per (problem, nvar)
instances = df[group_key].drop_duplicates()

# shuffle instances
instances = instances.sample(frac=1, random_state=66).reset_index(drop=True)

n = len(instances)
n_train = int(0.8 * n)

train_inst = instances.iloc[:n_train]
test_inst  = instances.iloc[n_train :]

# assign rows to splits by (problem, nvar)
train_df = df.merge(train_inst, on=["nvar", "name"], how="inner").reset_index(drop=True)
test_df  = df.merge(test_inst,  on=["nvar", "name"], how="inner").reset_index(drop=True)
all_df = pd.concat([train_df, test_df], axis=0)
all_df = all_df.reset_index(drop=True)
all_df

,nvar,extract_duration_problem,alloc_obj,total_alloc,time_grad,count_trigonometric_function,count_plus_minus,mem,neval_obj,alloc_grad,name,count_op_function,time_obj,timed_bytes,neval_grad,highest_degree (ExprTree),stats_elapsed_time,tree_length,tree_depth,count_exponential_function,mem_count
0,100,4.309682,34515160.0,77126264.0,0.464686,0.0,305.0,1,4.0,42611104.0,arglina,509.0,0.376715,18576.0,4.0,2.0,0.000135,10404.0,9.0,0.0,100
1,100,4.309682,34515160.0,77126264.0,0.464686,0.0,305.0,2,2.0,42611104.0,arglina,509.0,0.376715,22120.0,2.0,2.0,0.000151,10404.0,9.0,0.0,100
2,100,4.309682,34515160.0,77126264.0,0.464686,0.0,305.0,3,2.0,42611104.0,arglina,509.0,0.376715,25592.0,2.0,2.0,0.000140,10404.0,9.0,0.0,100
3,100,4.309682,34515160.0,77126264.0,0.464686,0.0,305.0,4,2.0,42611104.0,arglina,509.0,0.376715,29128.0,2.0,2.0,0.000151,10404.0,9.0,0.0,100
4,100,4.309682,34515160.0,77126264.0,0.464686,0.0,305.0,5,2.0,42611104.0,arglina,509.0,0.376715,32600.0,2.0,2.0,0.000139,10404.0,9.0,0.0,100
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
21495,10000,74.643566,75185144.0,546412448.0,1.054803,0.0,27501.0,96,68.0,471227304.0,woods,57501.0,0.113105,31484720.0,59.0,2.0,7.757891,42500.0,7.0,0.0,100
21496,10000,74.643566,75185144.0,546412448.0,1.054803,0.0,27501.0,97,68.0,471227304.0,woods,57501.0,0.113105,31804864.0,59.0,2.0,7.755893,42500.0,7.0,0.0,100
21497,10000,74.643566,75185144.0,546412448.0,1.054803,0.0,27501.0,98,68.0,471227304.0,woods,57501.0,0.113105,32125008.0,59.0,2.0,7.753765,42500.0,7.0,0.0,100
21498,10000,74.643566,75185144.0,546412448.0,1.054803,0.0,27501.0,99,68.0,471227304.0,woods,57501.0,0.113105,32445152.0,59.0,2.0,7.755922,42500.0,7.0,0.0,100


In [428]:
 # confirm and use the three predicted feature to predict time                                                                                                              #  
X_train = train_df[feature_cols_reg].to_numpy(dtype=float)
X_test  = test_df[feature_cols_reg].to_numpy(dtype=float)

y_train = np.log1p(train_df[target_cols_reg].to_numpy(dtype=float))
y_test  = np.log1p(test_df[target_cols_reg].to_numpy(dtype=float))

X_all = np.concatenate([X_train, X_test], axis=0)


### Load Stage 1 RF and GB and form the Best Stage 1 Hybrid Model

In [429]:
rf_stage1 = joblib.load('./model_weights/best_rf_model_reg.pkl')
gb_stage1 = joblib.load('./model_weights/best_xgb_model_reg.pkl')

In [430]:
oof_train_rf = cross_val_predict(rf_stage1, X_train, y_train, cv=5, n_jobs=-1)
oof_train_gb = cross_val_predict(gb_stage1, X_train, y_train, cv=5, n_jobs=-1)

# Test set predictions are generated normally using the fully-fitted models
test_pred_rf = rf_stage1.predict(X_test)
test_pred_gb = gb_stage1.predict(X_test)

# Stack features in pure LOG-SPACE
X_train_time = np.column_stack([
    oof_train_rf[:, 0], 
    oof_train_rf[:, 1], 
    oof_train_gb[:, 2]
])

X_test_time = np.column_stack([
    test_pred_rf[:, 0], 
    test_pred_rf[:, 1], 
    test_pred_gb[:, 2]
])

In [431]:
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

### Performance Ratio Global Function

In [376]:
import numpy as np
import pandas as pd
from scipy.stats import gmean

def evaluate_model(model, label="Model"):
    """
    Evaluates a stage-2 model using global X_test_time and global test_df.
    Includes arithmetic mean, geometric mean, and maximum performance ratio metrics.
    """
    predictions = model.predict(X_test_time)
    working_df = test_df.copy() 
    
    working_df["predicted_time"] = predictions
    group_cols = ["name", "nvar"]

    # 1. Model Selection
    model_choices = (
        working_df.sort_values(by=group_cols + ["predicted_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "predicted_time", "stats_elapsed_time"]]
        .rename(columns={"mem": "predicted_mem", "stats_elapsed_time": "actual_time"})
    )

    # 2. Oracle Selection
    oracle_choices = (
        working_df.sort_values(by=group_cols + ["stats_elapsed_time", "mem"], ascending=[True, True, True, False])
        .groupby(group_cols).first().reset_index()
        [group_cols + ["mem", "stats_elapsed_time"]]
        .rename(columns={"mem": "best_mem", "stats_elapsed_time": "best_time"})
    )

    # 3. Merge Matrix & Apply Hardened 0.10s Absolute Tolerance Filter
    final_df = pd.merge(model_choices, oracle_choices, on=group_cols).copy()
    final_df['actual_time'] = pd.to_numeric(final_df['actual_time'], errors='coerce')
    final_df['best_time'] = pd.to_numeric(final_df['best_time'], errors='coerce')

    final_df['raw_ratio'] = final_df['actual_time'] / final_df['best_time']
    time_diff = np.abs(final_df['actual_time'] - final_df['best_time'])
    final_df['performance_ratio'] = np.where(time_diff <= 0.10, 1.0, final_df['raw_ratio'])

    # Reorder display columns
    column_order = group_cols + ["predicted_mem", "predicted_time", "actual_time", "best_mem", "best_time", "performance_ratio", "raw_ratio"]
    final_df = final_df[column_order]

    # Output Summary Blocks
    print("\n" + "="*145)
    print(f"PER-PROBLEM PERFORMANCE PROFILE SUMMARY ({label.upper()})")
    print("="*145)
    print(final_df.to_string(index=False, formatters={
        'predicted_time': '{:.4f}'.format, 'actual_time': '{:.4f}'.format,
        'best_time': '{:.4f}'.format, 'performance_ratio': '{:.4f}'.format, 'raw_ratio': '{:.4f}'.format
    }))
    print("="*145)

    total_actual = final_df['actual_time'].sum()
    total_best = final_df['best_time'].sum()
    
    # Calculate Core Evaluation Statistics
    global_mean_ratio = final_df['performance_ratio'].mean()
    geom_mean_ratio = gmean(final_df['performance_ratio'])  # Clean Scipy implementation
    max_performance_ratio = final_df['performance_ratio'].max()
    
    print("\n" + "="*50)
    print(f"AGGREGATE METRICS ({label.upper()})")
    print("="*50)
    print(final_df['performance_ratio'].quantile([0.50, 0.75, 0.90]).to_string(float_format='{:.4f}'.format))
    print("-"*50)
    print(f"Global Mean Perf. Ratio     : {global_mean_ratio:.4f}")
    print(f"Geometric Mean Perf. Ratio  : {geom_mean_ratio:.4f}")
    print(f"Maximum Performance Ratio   : {max_performance_ratio:.4f}")
    print("-"*50)
    print(f"Total Actual Compute Time   : {total_actual:.4f}s")
    print(f"Total Oracle Optimal Time   : {total_best:.4f}s")
    print("="*50)
    
    return final_df

## Train Model directly on Best Stage 1 Hybrid Model Predicted Features

### Using Predicted Data from Best Stage 1 Hybrid Model For Linear Regression

In [432]:
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import cross_val_predict
import joblib  # Standard library for saving scikit-learn models

In [396]:
# Initialize Stage 2 Linear Regression Regressor
lr_predictor = LinearRegression()

# 1. 5-Fold Cross-Validation Step (on Training Data in log-space)
cv_pred_log = cross_val_predict(lr_predictor, X_train_time, y_train_time, cv=5)
cv_mse = mean_squared_error(y_train_time, cv_pred_log)
cv_r2 = r2_score(y_train_time, cv_pred_log)

print("--- 5-Fold Cross-Validation Metrics (Train Set - Log Scale) ---")
print(f"CV MSE = {cv_mse:.4f} | CV R² = {cv_r2:.4f}")

# 2. Final Fit and Test Prediction
lr_predictor.fit(X_train_time, y_train_time)
pred_log = lr_predictor.predict(X_test_time)

# 3. Inverse-Transform to Original Physical Scale for Evaluation Metrics
# y_test_orig = np.expm1(y_test_time)
# pred_orig = np.expm1(pred_log)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")

--- 5-Fold Cross-Validation Metrics (Train Set - Log Scale) ---
CV MSE = 2.1330 | CV R² = 0.6868

--- Final Test Set Metrics (Original Physical Scale) ---
MSE = 1.8783 | R² = 0.5841

--- Quantiled Relative Error Q(q) (Test Set) ---
['stats_elapsed_time']:
Quantile (q) | Relative Error Value Q(q)
  Q(0.25):  0.3517
  Q(0.50):  0.8962
  Q(0.75):  12.4978


In [397]:
evaluate_model(lr_predictor)


PER-PROBLEM PERFORMANCE PROFILE SUMMARY (MODEL)
        name   nvar  predicted_mem predicted_time actual_time  best_mem best_time performance_ratio raw_ratio
     arglinb    100              1        -0.5404      0.0002         1    0.0002            1.0000    1.0000
     argtrig    100              1        -0.9216      0.0135        61    0.0081            1.0000    1.6555
     arwhead   1000              1         0.8167      0.0589         3    0.0401            1.0000    1.4710
     brownal    100              1        -0.5861      0.0003         1    0.0003            1.0000    1.0000
      brybnd  10000              3         3.2076     24.3856         1   23.0197            1.0593    1.0593
    clplatea    961              3         0.7494      4.1646        97    1.0576            3.9377    3.9377
    clplatec    100              1        -0.6780      0.9991        87    0.0129           77.6369   77.6369
      cosine    100              1        -0.9879      0.0006         1

,name,nvar,predicted_mem,predicted_time,actual_time,best_mem,best_time,performance_ratio,raw_ratio
0,arglinb,100,1,-0.540376,0.000229,1,0.000229,1.000000,1.000000
1,argtrig,100,1,-0.921597,0.013453,61,0.008126,1.000000,1.655547
2,arwhead,1000,1,0.816713,0.058931,3,0.040062,1.000000,1.470991
3,brownal,100,1,-0.586143,0.000280,1,0.000280,1.000000,1.000000
4,brybnd,10000,3,3.207568,24.385600,1,23.019709,1.059336,1.059336
5,clplatea,961,3,0.749397,4.164609,97,1.057629,3.937685,3.937685
6,clplatec,100,1,-0.677958,0.999100,87,0.012869,77.636894,77.636894
7,cosine,100,1,-0.987855,0.000634,1,0.000634,1.000000,1.000000
8,cragglvy,1000,1,1.003427,0.415430,12,0.247496,1.678532,1.678532
9,cragglvy,10000,3,2.564184,27.082774,6,24.499405,1.105446,1.105446


### Using Predicted Data from Best Stage 1 Hybrid Model For Random Forest

In [ ]:
# import numpy as np
# import pandas as pd
# from scipy.stats import randint, uniform
# from sklearn.ensemble import RandomForestRegressor
# from sklearn.model_selection import RandomizedSearchCV

# # ==============================================================================
# # SEARCH SPACE CONFIGURATION
# # ==============================================================================
# param_distributions = {
#     'n_estimators': randint(100, 300),
#     'max_depth': [None, 10, 20, 30],
#     'min_samples_split': randint(2, 10),
#     'min_samples_leaf': randint(1, 5),
#     'max_features': [0.5, 0.7, 'sqrt'],
#     'bootstrap': [True],
#     'min_impurity_decrease': uniform(0, 0.001), 
#     'ccp_alpha': uniform(0, 0.001),             
#     'criterion': ['squared_error', 'friedman_mse']
# }

# rf_stage2 = RandomForestRegressor(random_state=42, n_jobs=1)

# # Set n_iter=1000 and verbose=3 to print real-time fold updates
# time_random_search = RandomizedSearchCV(
#     rf_stage2,
#     param_distributions=param_distributions,
#     n_iter=1000,
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=4,
#     random_state=42,
#     verbose=3,
#     return_train_score=True,
#     error_score='raise'
# )

# print("Starting 1,000-iteration Randomized Search Cross-Validation...")
# print(f"Total Fits to Execute: {1000 * 5} (1000 iterations x 5 folds)\n")

# # Fit Stage 2 on log features to predict your log targets
# time_random_search.fit(X_train_time, y_train_time)

# # ==============================================================================
# # PROGRESSION LANDSCAPE ANALYSIS (Print Data Along the Way)
# # ==============================================================================
# results_df = pd.DataFrame(time_random_search.cv_results_)
# results_df['actual_mse'] = -results_df['mean_test_score']

# print("\n" + "="*80)
# print("RANDOM SEARCH SEARCH-SPACE LANDSCAPE (MILESTONE INTERVALS)")
# print("="*80)
# print("Checking performance distribution across the 1,000 runs:")

# # Print structural metrics at regular intervals across the search trajectory
# intervals = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
# milestone_cols = ['actual_mse', 'mean_fit_time', 'param_n_estimators', 
#                   'param_max_depth', 'param_min_samples_split', 'param_min_samples_leaf']

# sorted_by_run = results_df.copy() # Keep sequential run order intact for tracking
# print(sorted_by_run[milestone_cols].iloc[intervals].to_string(index=True))

# print("\n" + "="*80)
# print("BEST HYPERPARAMETERS FOUND")
# print("="*80)
# print(f"Best Parameters : {time_random_search.best_params_}")
# print(f"Best Log-MSE    : {-time_random_search.best_score_:.4f}")
# print("="*80)

# # Export full log profile for deep investigation
# results_df.to_csv('rf_stage2_1000runs_trajectory.csv', index=False)

Starting 1,000-iteration Randomized Search Cross-Validation...
Total Fits to Execute: 5000 (1000 iterations x 5 folds)

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
[CV 2/5] END bootstrap=True, ccp_alpha=0.0003745401188473625, criterion=squared_error, max_depth=20, max_features=sqrt, min_impurity_decrease=0.0007796910002727693, min_samples_leaf=1, min_samples_split=8, n_estimators=221;, score=(train=-0.171, test=-2.842) total time=   1.3s
[CV 3/5] END bootstrap=True, ccp_alpha=0.0003745401188473625, criterion=squared_error, max_depth=20, max_features=sqrt, min_impurity_decrease=0.0007796910002727693, min_samples_leaf=1, min_samples_split=8, n_estimators=221;, score=(train=-0.182, test=-2.531) total time=   1.3s
[CV 4/5] END bootstrap=True, ccp_alpha=0.0003745401188473625, criterion=squared_error, max_depth=20, max_features=sqrt, min_impurity_decrease=0.0007796910002727693, min_samples_leaf=1, min_samples_split=8, n_estimators=221;, score=(train=-0.201, test=-1.869) 

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/numpy/ma/core.py:2820: RuntimeWarning: invalid value encountered in cast
  _data = np.array(data, dtype=dtype, copy=copy,



RANDOM SEARCH SEARCH-SPACE LANDSCAPE (MILESTONE INTERVALS)
Checking performance distribution across the 1,000 runs:
     actual_mse  mean_fit_time  param_n_estimators param_max_depth  param_min_samples_split  param_min_samples_leaf
0      2.739877       1.234382                 221              20                        8                       1
100    3.232085       1.662720                 103            None                        6                       1
200    2.618096       1.413217                 257              10                        9                       2
300    3.149204       1.447265                 141              20                        7                       2
400    2.539607       1.113225                 218              10                        6                       3
500    3.136640       3.013652                 202            None                        9                       2
600    2.660830       1.661373                 295              20     

In [398]:
time_best_model_rf = joblib.load('./model_weights/best_hybrid_rf_model_time_new_cv.pkl')
# time_best_model_rf = time_random_search.best_estimator_
accuracy = time_best_model_rf.score(X_test_time, y_test_time)
print(f"Time prediction accuracy based on predicted values={accuracy}")
# joblib.dump(time_best_model_rf, './model_weights/best_hybrid_rf_model_time_new_cv.pkl')
# joblib.dump(time_best_model_rf, './model_weights/best_hybrid_rf_model_time.pkl')

Time prediction accuracy based on predicted values=0.6236072929946244


In [399]:
pred = time_best_model_rf.predict(X_test_time)
test_mse = mean_squared_error(y_test_time, pred)
test_r2  = r2_score(y_test_time, pred)
print(f"MSE={test_mse:.4f}  R²={test_r2:.4f}")

# Compute pointwise relative error matching the paper's formula: |y - y_hat| / (|y| + delta)
relative_error = np.abs(pred - y_test_time) / (np.abs(y_test_time) + 1e-8)

# Quantiled Relative Error Distribution
quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_col_model):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    
    # Safe extraction whether relative_error is a 1D vector or a 2D column matrix
    error_data = relative_error[:, i] if relative_error.ndim > 1 else relative_error
    q_values = np.quantile(error_data, quantiles)
    
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")
print(list(pred))

MSE=1.6999  R²=0.6236

--- Quantiled Relative Error Q(q) (Test Set) ---

stats_elapsed_time:
Quantile (q) | Relative Error Value Q(q)
  Q(0.25):  0.3084
  Q(0.50):  0.7746
  Q(0.75):  6.2418
[0.22065154357239058, 0.19431057285621803, 0.0636602605177706, 0.08533355910223044, 0.0704865907080243, 0.12353962273276282, 0.12480906497717613, 0.12353962273276282, 0.12597580327970356, 0.11201375054942032, 0.12353962273276282, 0.12419322852176849, 0.13855352822027642, 0.1500029748400398, 0.15083135650900303, 0.1532617341068535, 0.18698860706769108, 0.18698860706769108, 0.2507587123624685, 0.1834280564040569, 0.19565395805366276, 0.15451832293058124, 0.18373084943177342, 0.19565395805366276, 0.1615829125285243, 0.23860997770640743, 0.2530611065586457, 0.3318313284385438, 0.3850038442824034, 0.4250358332297291, 0.4149167015703845, 0.5974154697272561, 0.7541386129301187, 0.6976995930130615, 0.7348412888248727, 0.7348412888248727, 0.8325851119355275, 0.8325703709945235, 0.8325703709945235, 0.8223115

In [400]:
evaluate_model(joblib.load('./model_weights/best_hybrid_rf_model_time_new_cv.pkl'))
# evaluate_model(joblib.load('./model_weights/best_hybrid_rf_model_time.pkl'))


PER-PROBLEM PERFORMANCE PROFILE SUMMARY (MODEL)
        name   nvar  predicted_mem predicted_time actual_time  best_mem best_time performance_ratio raw_ratio
     arglinb    100              3         0.0637      0.0003         1    0.0002            1.0000    1.2844
     argtrig    100             50         0.0469      0.0100        61    0.0081            1.0000    1.2303
     arwhead   1000             47         0.3842      0.0461         3    0.0401            1.0000    1.1508
     brownal    100              7         0.0993      0.0003         1    0.0003            1.0000    1.0928
      brybnd  10000             20         2.1251     24.3788         1   23.0197            1.0590    1.0590
    clplatea    961              8         0.4257      2.1627        97    1.0576            2.0449    2.0449
    clplatec    100            100         0.0755      0.0208        87    0.0129            1.0000    1.6133
      cosine    100             29         0.0880      0.0010         1

,name,nvar,predicted_mem,predicted_time,actual_time,best_mem,best_time,performance_ratio,raw_ratio
0,arglinb,100,3,0.063660,0.000294,1,0.000229,1.000000,1.284375
1,argtrig,100,50,0.046949,0.009998,61,0.008126,1.000000,1.230349
2,arwhead,1000,47,0.384161,0.046104,3,0.040062,1.000000,1.150810
3,brownal,100,7,0.099257,0.000306,1,0.000280,1.000000,1.092845
4,brybnd,10000,20,2.125120,24.378822,1,23.019709,1.059041,1.059041
5,clplatea,961,8,0.425735,2.162715,97,1.057629,2.044871,2.044871
6,clplatec,100,100,0.075480,0.020761,87,0.012869,1.000000,1.613273
7,cosine,100,29,0.087950,0.001018,1,0.000634,1.000000,1.605867
8,cragglvy,1000,51,0.299760,0.328729,12,0.247496,1.000000,1.328218
9,cragglvy,10000,1,3.642981,31.816462,6,24.499405,1.298663,1.298663


### Using Predicted Data from Best Stage 1 Hybrid Model For XGBoost

In [ ]:
# import numpy as np
# import pandas as pd
# import xgboost as xgb
# from scipy.stats import randint, uniform
# from sklearn.model_selection import RandomizedSearchCV

# # ==============================================================================
# # ROBUST SEARCH SPACE FOR LOG-SPACE META-MODELING
# # ==============================================================================
# param_distributions = {
#     'n_estimators':      randint(100, 500),
#     'learning_rate':     uniform(0.01, 0.15),  # Lower, stable rates to prevent variance spikes
#     'max_depth':         randint(2, 6),        # Shallow trees prevent overfitting to Stage 1 noise
#     'subsample':         uniform(0.6, 0.4),    # Stochastic sampling for generalization
#     'colsample_bytree':  uniform(0.6, 0.4),    
#     'min_child_weight':  randint(1, 10),
    
#     # Tightened structural constraints to match low-variance log targets
#     'gamma':             uniform(0, 0.1),      
#     'reg_alpha':         uniform(0, 0.1),      # L1 Regularization
#     'reg_lambda':        uniform(0.5, 1.5),    # L2 Regularization
# }

# # Utilizing hist tree method for extreme speed optimization across 5,000 total fits
# xgb_model = xgb.XGBRegressor(tree_method='hist', random_state=42, n_jobs=1)

# time_random_search = RandomizedSearchCV(
#     xgb_model,
#     param_distributions=param_distributions,
#     n_iter=1000, # 1,000 iterations across the CV loop
#     cv=5,
#     scoring='neg_mean_squared_error',
#     n_jobs=4,
#     random_state=42,
#     verbose=3,
#     return_train_score=True,
#     error_score='raise'
# )

# print("Starting 1,000-iteration Robust XGBoost Stage 2 Search...")
# print(f"Total Fits to Execute: {1000 * 5} (1000 iterations x 5 folds)\n")

# # Fit the robust model on your un-leaked out-of-fold log feature matrix
# time_random_search.fit(X_train_time, y_train_time)

# # ==============================================================================
# # PROGRESSION TRAJECTORY ANALYSIS (Print Data Along the Way)
# # ==============================================================================
# results_df = pd.DataFrame(time_random_search.cv_results_)
# results_df['actual_mse'] = -results_df['mean_test_score']

# print("\n" + "="*90)
# print("XGBOOST SEARCH-SPACE TRAJECTORY LANDSCAPE (MILESTONE INTERVALS)")
# print("="*90)
# print("Tracking hyperparameter convergence across sequential intervals:")

# # Extract performance indicators at regular milestones across the 1000 runs
# intervals = [0, 100, 200, 300, 400, 500, 600, 700, 800, 900, 999]
# milestone_cols = ['actual_mse', 'mean_fit_time', 'param_n_estimators', 
#                   'param_learning_rate', 'param_max_depth', 'param_subsample',
#                   'param_reg_alpha', 'param_reg_lambda']

# sorted_by_run = results_df.copy() # Keeps the initial processing timeline visible
# print(sorted_by_run[milestone_cols].iloc[intervals].to_string(index=True))

# print("\n" + "="*80)
# print("OPTIMAL CONFIGURATION METRICS")
# print("="*80)
# print(f"Best Parameters : {time_random_search.best_params_}")
# print(f"Best Log-MSE    : {-time_random_search.best_score_:.4f}")
# print("="*80)

# # Export complete trajectory framework for performance logging
# results_df.to_csv('xgb_randomsearch_stage2_1000runs_trajectory.csv', index=False)

Starting 1,000-iteration Robust XGBoost Stage 2 Search...
Total Fits to Execute: 5000 (1000 iterations x 5 folds)

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits
[CV 1/5] END colsample_bytree=0.749816047538945, gamma=0.09507143064099162, learning_rate=0.11979909127171076, max_depth=2, min_child_weight=5, n_estimators=202, reg_alpha=0.04458327528535912, reg_lambda=0.6499623737270044, subsample=0.7836995567863468;, score=(train=-0.932, test=-1.477) total time=   0.1s
[CV 2/5] END colsample_bytree=0.749816047538945, gamma=0.09507143064099162, learning_rate=0.11979909127171076, max_depth=2, min_child_weight=5, n_estimators=202, reg_alpha=0.04458327528535912, reg_lambda=0.6499623737270044, subsample=0.7836995567863468;, score=(train=-0.778, test=-2.792) total time=   0.1s
[CV 3/5] END colsample_bytree=0.749816047538945, gamma=0.09507143064099162, learning_rate=0.11979909127171076, max_depth=2, min_child_weight=5, n_estimators=202, reg_alpha=0.04458327528535912, reg_lambda=

In [383]:
time_best_model_gb = joblib.load('./model_weights/best_hybrid_xgb_model_time_new_cv.pkl')
# time_best_model_gb = time_random_search.best_estimator_
accuracy = time_best_model_gb.score(X_test_time, y_test_time)
print(f"Time prediction accuracy based on predicted values={accuracy}")

# joblib.dump(time_best_model_gb, './model_weights/best_hybrid_xgb_model_time_new_cv.pkl')

Time prediction accuracy based on predicted values=0.5659815122354326


In [384]:
pred = time_best_model_gb.predict(X_test_time)
test_mse = mean_squared_error(y_test_time, pred)
test_r2  = r2_score(y_test_time, pred)
print(f"MSE={test_mse:.4f}  R²={test_r2:.4f}")

# Compute pointwise relative error matching the paper's formula: |y - y_hat| / (|y| + delta)
relative_error = np.abs(pred - y_test_time) / (np.abs(y_test_time) + 1e-8)

# Quantiled Relative Error Distribution
quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
for i, col in enumerate(target_col_model):
    print(f"\n{col}:")
    print(f"Quantile (q) | Relative Error Value Q(q)")
    
    # Safe extraction whether relative_error is a 1D vector or a 2D column matrix
    error_data = relative_error[:, i] if relative_error.ndim > 1 else relative_error
    q_values = np.quantile(error_data, quantiles)
    
    for q, val in zip(quantiles, q_values):
        print(f"  Q({q:.2f}):  {val:.4f}")

MSE=1.9602  R²=0.5660

--- Quantiled Relative Error Q(q) (Test Set) ---

stats_elapsed_time:
Quantile (q) | Relative Error Value Q(q)
  Q(0.25):  0.4016
  Q(0.50):  0.8059
  Q(0.75):  18.4072


In [385]:
evaluate_model(joblib.load('./model_weights/best_hybrid_xgb_model_time_new_cv.pkl'))


PER-PROBLEM PERFORMANCE PROFILE SUMMARY (MODEL)
        name   nvar  predicted_mem predicted_time actual_time  best_mem best_time performance_ratio raw_ratio
     arglinb    100              2         0.0820      0.0003         1    0.0002            1.0000    1.2844
     argtrig    100             70         0.0820      0.0180        61    0.0081            1.0000    2.2172
     arwhead   1000              3         0.7408      0.0401         3    0.0401            1.0000    1.0000
     brownal    100              1         0.2182      0.0003         1    0.0003            1.0000    1.0000
      brybnd  10000             11         3.5098     24.3659         1   23.0197            1.0585    1.0585
    clplatea    961             12         0.7615      2.0763        97    1.0576            1.9631    1.9631
    clplatec    100             67        -0.6387      0.0177        87    0.0129            1.0000    1.3763
      cosine    100              1        -0.1049      0.0006         1

,name,nvar,predicted_mem,predicted_time,actual_time,best_mem,best_time,performance_ratio,raw_ratio
0,arglinb,100,2,0.081954,0.000294,1,0.000229,1.000000,1.284375
1,argtrig,100,70,0.081954,0.018017,61,0.008126,1.000000,2.217205
2,arwhead,1000,3,0.740786,0.040062,3,0.040062,1.000000,1.000000
3,brownal,100,1,0.218215,0.000280,1,0.000280,1.000000,1.000000
4,brybnd,10000,11,3.509773,24.365939,1,23.019709,1.058482,1.058482
5,clplatea,961,12,0.761547,2.076262,97,1.057629,1.963129,1.963129
6,clplatec,100,67,-0.638743,0.017711,87,0.012869,1.000000,1.376260
7,cosine,100,1,-0.104909,0.000634,1,0.000634,1.000000,1.000000
8,cragglvy,1000,3,0.826571,0.310437,12,0.247496,1.000000,1.254311
9,cragglvy,10000,3,1.811823,27.082774,6,24.499405,1.105446,1.105446


## Use Pre-trained Stage 2 Models From Our Empirical Dataset directly to predict time from Best Stage 1 Hybrid Predicted Features

In [433]:
y_train_time = np.log1p(train_df[target_col_model].to_numpy(dtype=float)).ravel()
y_test_time = np.log1p(test_df[target_col_model].to_numpy(dtype=float)).ravel()

### Try Pre-Trained Stage 2 Linear Regression Model

In [434]:
# Load the saved model
lr_stage2 = joblib.load("./model_weights/best_lr_model_time.pkl")

# Use the LOADED model instance to predict
pred_log = lr_stage2.predict(X_test_time)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")


--- Final Test Set Metrics (Original Physical Scale) ---
MSE = 4.5620 | R² = -0.0101

--- Quantiled Relative Error Q(q) (Test Set) ---
['stats_elapsed_time']:
Quantile (q) | Relative Error Value Q(q)
  Q(0.25):  0.5209
  Q(0.50):  1.4777
  Q(0.75):  54.0398


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


In [404]:
evaluate_model(joblib.load("./model_weights/best_lr_model_time.pkl"))


PER-PROBLEM PERFORMANCE PROFILE SUMMARY (MODEL)
        name   nvar  predicted_mem predicted_time actual_time  best_mem best_time performance_ratio raw_ratio
     arglinb    100              1         2.0109      0.0002         1    0.0002            1.0000    1.0000
     argtrig    100             38         2.0108      0.0104        61    0.0081            1.0000    1.2835
     arwhead   1000             69         2.0109      0.0460         3    0.0401            1.0000    1.1473
     brownal    100              1         2.0109      0.0003         1    0.0003            1.0000    1.0000
      brybnd  10000              1         2.0109     23.0197         1   23.0197            1.0000    1.0000
    clplatea    961              8         2.0108      2.1627        97    1.0576            2.0449    2.0449
    clplatec    100             46         2.0108      0.0242        87    0.0129            1.0000    1.8779
      cosine    100             40         2.0108      0.0010         1

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


,name,nvar,predicted_mem,predicted_time,actual_time,best_mem,best_time,performance_ratio,raw_ratio
0,arglinb,100,1,2.010929,0.000229,1,0.000229,1.000000,1.000000
1,argtrig,100,38,2.010849,0.010430,61,0.008126,1.000000,1.283543
2,arwhead,1000,69,2.010870,0.045964,3,0.040062,1.000000,1.147316
3,brownal,100,1,2.010898,0.000280,1,0.000280,1.000000,1.000000
4,brybnd,10000,1,2.010871,23.019709,1,23.019709,1.000000,1.000000
5,clplatea,961,8,2.010831,2.162715,97,1.057629,2.044871,2.044871
6,clplatec,100,46,2.010843,0.024167,87,0.012869,1.000000,1.877946
7,cosine,100,40,2.010846,0.001026,1,0.000634,1.000000,1.618278
8,cragglvy,1000,36,2.010849,0.319299,12,0.247496,1.000000,1.290117
9,cragglvy,10000,36,2.010843,34.876608,6,24.499405,1.423570,1.423570


### Try Pre-Trained Stage 2 Random Forest

In [435]:
# Load the saved model
rf_stage2 = joblib.load("./model_weights/best_rf_model_time_raw.pkl")

# Use the LOADED model instance to predict
pred_log = rf_stage2.predict(X_test_time)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")


--- Final Test Set Metrics (Original Physical Scale) ---
MSE = 4.4933 | R² = 0.0051

--- Quantiled Relative Error Q(q) (Test Set) ---
['stats_elapsed_time']:
Quantile (q) | Relative Error Value Q(q)
  Q(0.25):  0.5080
  Q(0.50):  2.1647
  Q(0.75):  52.4424


/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


In [406]:
evaluate_model(joblib.load("./model_weights/best_rf_model_time_raw.pkl"))


PER-PROBLEM PERFORMANCE PROFILE SUMMARY (MODEL)
        name   nvar  predicted_mem predicted_time actual_time  best_mem best_time performance_ratio raw_ratio
     arglinb    100            100         1.8372      0.0003         1    0.0002            1.0000    1.2667
     argtrig    100            100         1.8450      0.0092        61    0.0081            1.0000    1.1280
     arwhead   1000            100         1.8450      0.0460         3    0.0401            1.0000    1.1480
     brownal    100            100         1.8372      0.0006         1    0.0003            1.0000    1.9787
      brybnd  10000              6         1.8372     24.3875         1   23.0197            1.0594    1.0594
    clplatea    961              9         2.5742      2.3199        97    1.0576            2.1935    2.1935
    clplatec    100            100         1.8372      0.0208        87    0.0129            1.0000    1.6133
      cosine    100            100         2.1125      0.0102         1

/Library/Frameworks/Python.framework/Versions/3.10/lib/python3.10/site-packages/sklearn/base.py:493: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(


,name,nvar,predicted_mem,predicted_time,actual_time,best_mem,best_time,performance_ratio,raw_ratio
0,arglinb,100,100,1.837173,0.000290,1,0.000229,1.000000,1.266667
1,argtrig,100,100,1.844965,0.009166,61,0.008126,1.000000,1.127982
2,arwhead,1000,100,1.844965,0.045992,3,0.040062,1.000000,1.148013
3,brownal,100,100,1.837173,0.000554,1,0.000280,1.000000,1.978705
4,brybnd,10000,6,1.837173,24.387542,1,23.019709,1.059420,1.059420
5,clplatea,961,9,2.574159,2.319937,97,1.057629,2.193527,2.193527
6,clplatec,100,100,1.837173,0.020761,87,0.012869,1.000000,1.613273
7,cosine,100,100,2.112496,0.010186,1,0.000634,1.000000,16.067319
8,cragglvy,1000,100,1.844965,0.338653,12,0.247496,1.000000,1.368317
9,cragglvy,10000,34,1.837173,33.130567,6,24.499405,1.352301,1.352301


### Try Pre-Trained Stage 2 Gradient Boosting

In [436]:
# Load the saved model
gb_stage2 = joblib.load("./model_weights/best_xgb_model_time_raw.pkl")

# Use the LOADED model instance to predict
pred_log = gb_stage2.predict(X_test_time)

test_mse = mean_squared_error(y_test_time, pred_log)
test_r2 = r2_score(y_test_time, pred_log)

print("\n--- Final Test Set Metrics (Original Physical Scale) ---")
print(f"MSE = {test_mse:.4f} | R² = {test_r2:.4f}")

# 4. Quantiled Relative Error Distribution (Original Scale)
relative_error = np.abs(pred_log - y_test_time) / (np.abs(y_test_time) + 1e-8)

quantiles = [0.25, 0.50, 0.75]
print("\n--- Quantiled Relative Error Q(q) (Test Set) ---")
print(f"{target_col_model}:")
print(f"Quantile (q) | Relative Error Value Q(q)")

q_values = np.quantile(relative_error, quantiles)
for q, val in zip(quantiles, q_values):
    print(f"  Q({q:.2f}):  {val:.4f}")


--- Final Test Set Metrics (Original Physical Scale) ---
MSE = 6.7783 | R² = -0.5008

--- Quantiled Relative Error Q(q) (Test Set) ---
['stats_elapsed_time']:
Quantile (q) | Relative Error Value Q(q)
  Q(0.25):  0.8476
  Q(0.50):  0.9538
  Q(0.75):  5.9993


In [408]:
evaluate_model(joblib.load("./model_weights/best_xgb_model_time_raw.pkl"))


PER-PROBLEM PERFORMANCE PROFILE SUMMARY (MODEL)
        name   nvar  predicted_mem predicted_time actual_time  best_mem best_time performance_ratio raw_ratio
     arglinb    100            100         0.2121      0.0003         1    0.0002            1.0000    1.2667
     argtrig    100            100         0.2121      0.0092        61    0.0081            1.0000    1.1280
     arwhead   1000            100         0.2121      0.0460         3    0.0401            1.0000    1.1480
     brownal    100            100         0.2121      0.0006         1    0.0003            1.0000    1.9787
      brybnd  10000            100         0.2121     24.4133         1   23.0197            1.0605    1.0605
    clplatea    961            100         0.7492      1.1928        97    1.0576            1.1278    1.1278
    clplatec    100            100         0.2121      0.0208        87    0.0129            1.0000    1.6133
      cosine    100            100         0.2121      0.0102         1

,name,nvar,predicted_mem,predicted_time,actual_time,best_mem,best_time,performance_ratio,raw_ratio
0,arglinb,100,100,0.212050,0.000290,1,0.000229,1.000000,1.266667
1,argtrig,100,100,0.212050,0.009166,61,0.008126,1.000000,1.127982
2,arwhead,1000,100,0.212050,0.045992,3,0.040062,1.000000,1.148013
3,brownal,100,100,0.212050,0.000554,1,0.000280,1.000000,1.978705
4,brybnd,10000,100,0.212050,24.413258,1,23.019709,1.060537,1.060537
5,clplatea,961,100,0.749200,1.192818,97,1.057629,1.127823,1.127823
6,clplatec,100,100,0.212050,0.020761,87,0.012869,1.000000,1.613273
7,cosine,100,100,0.212050,0.010186,1,0.000634,1.000000,16.067319
8,cragglvy,1000,100,0.212050,0.338653,12,0.247496,1.000000,1.368317
9,cragglvy,10000,100,0.212050,37.068113,6,24.499405,1.513021,1.513021


## Baseline Performance

In [262]:
import pandas as pd

print("--------------------------------------------------------------------\n"
      "Performance For Static Heuristic Baselines (Mem=5 vs. Mem=87)")

group_cols = ["name", "nvar"]

# 1. Oracle Selection (Heuristic: Min actual time, break ties with LARGEST mem)
oracle_choices = (
    test_df.sort_values(by=group_cols + ["stats_elapsed_time", "mem"], ascending=[True, True, True, False])
    .groupby(group_cols).first().reset_index()
    [group_cols + ["mem", "stats_elapsed_time"]]
    .rename(columns={"mem": "best_mem", "stats_elapsed_time": "best_time"})
)

# 2. Extract actual recorded times when mem = 5 (Baseline 1: JSOSolver default)
baseline_5 = (
    test_df[test_df["mem"] == 5][group_cols + ["stats_elapsed_time"]]
    .rename(columns={"stats_elapsed_time": "time_mem5"})
)

# 3. Extract actual recorded times when mem = 87 (Baseline 2: Global Best)
baseline_87 = (
    test_df[test_df["mem"] == 87][group_cols + ["stats_elapsed_time"]]
    .rename(columns={"stats_elapsed_time": "time_mem87"})
)

# 4. Merge baseline profiles into a unified comparison matrix
comparison_df = pd.merge(oracle_choices, baseline_5, on=group_cols, how="left")
comparison_df = pd.merge(comparison_df, baseline_87, on=group_cols, how="left")

# 5. Calculate individual performance ratios (Actual Baseline Time / True Oracle Time)
comparison_df["ratio_mem5"] = comparison_df["time_mem5"] / comparison_df["best_time"]
comparison_df["ratio_mem87"] = comparison_df["time_mem87"] / comparison_df["best_time"]

# Reorder columns for scannability
print_cols = group_cols + ["best_mem", "best_time", "time_mem5", "ratio_mem5", "time_mem87", "ratio_mem87"]
final_baseline_df = comparison_df[print_cols]

# ==============================================================================
# VISUALIZATION & OUTPUT
# ==============================================================================

# 1. Output granular per-problem table
print("\n" + "="*145)
print("PER-PROBLEM STATIC BASELINE SIDE-BY-SIDE COMPARISON")
print("="*145)
print(final_baseline_df.to_string(index=False, formatters={
    'best_time': '{:.4f}'.format,
    'time_mem5': '{:.4f}'.format,
    'ratio_mem5': '{:.4f}'.format,
    'time_mem87': '{:.4f}'.format,
    'ratio_mem87': '{:.4f}'.format
}))
print("="*145)

# 2. Compute aggregate metrics and Global Performance Ratios
quantiles = [0.25, 0.50, 0.75, 0.90]
summary_df = pd.DataFrame({
    "Quantile (q)": quantiles,
    "Baseline 1 (mem=5)": final_baseline_df["ratio_mem5"].quantile(quantiles).values,
    "Baseline 2 (mem=87)": final_baseline_df["ratio_mem87"].quantile(quantiles).values
})

total_best_time = final_baseline_df["best_time"].sum()
total_time_mem5 = final_baseline_df["time_mem5"].sum()
total_time_mem87 = final_baseline_df["time_mem87"].sum()

global_ratio_mem5 = total_time_mem5 / total_best_time
global_ratio_mem87 = total_time_mem87 / total_best_time

print("\n" + "="*70)
print("AGGREGATE BASELINE PERFORMANCE STATISTICS")
print("="*70)
print(summary_df.to_string(index=False, formatters={
    'Baseline 1 (mem=5)': '{:.4f}'.format,
    'Baseline 2 (mem=87)': '{:.4f}'.format
}))
print("-"*70)
print(f"Total Compute Time (mem=5)        : {total_time_mem5:.4f}s")
print(f"Total Compute Time (mem=87)       : {total_time_mem87:.4f}s")
print(f"Total Oracle Optimal Time         : {total_best_time:.4f}s")
print("-"*70)
print(f"GLOBAL PERFORMANCE RATIO (mem=5)  : {global_ratio_mem5:.4f}")
print(f"GLOBAL PERFORMANCE RATIO (mem=87) : {global_ratio_mem87:.4f}")
print("="*70)

--------------------------------------------------------------------
Performance For Static Heuristic Baselines (Mem=5 vs. Mem=87)

PER-PROBLEM STATIC BASELINE SIDE-BY-SIDE COMPARISON
        name   nvar  best_mem best_time time_mem5 ratio_mem5 time_mem87 ratio_mem87
     arglinb    100         1    0.0002    0.0002     1.0229     0.0003      1.2625
     argtrig    100        61    0.0081    0.0112     1.3724     0.0117      1.4379
     arwhead   1000         3    0.0401    0.0403     1.0068     0.0460      1.1474
     brownal    100         1    0.0003    0.0004     1.5009     0.0006      1.9761
      brybnd  10000         1   23.0197   24.3815     1.0592    24.3938      1.0597
    clplatea    961        97    1.0576    2.9448     2.7843     1.3719      1.2972
    clplatec    100        87    0.0129    0.4126    32.0606     0.0129      1.0000
      cosine    100         1    0.0006    0.0011     1.6988     0.0012      1.8879
    cragglvy   1000        12    0.2475    0.3020     1.2202

In [393]:
class StaticBaselineModel:
    def __init__(self, target_mem):
        self.target_mem = target_mem
        
    def predict(self, X):
        # Force the selection logic to always pick the target memory tier
        return np.where(test_df["mem"] == self.target_mem, 0.0, 9999.0)

# ==============================================================================
# UNIFIED BASELINE EVALUATION RUNS
# ==============================================================================
evaluate_model(StaticBaselineModel(5), label="Baseline 1 (Mem=5)")
evaluate_model(StaticBaselineModel(87), label="Baseline 2 (Mem=87)")


PER-PROBLEM PERFORMANCE PROFILE SUMMARY (BASELINE 1 (MEM=5))
        name   nvar  predicted_mem predicted_time actual_time  best_mem best_time performance_ratio raw_ratio
     arglinb    100              5         0.0000      0.0002         1    0.0002            1.0000    1.0229
     argtrig    100              5         0.0000      0.0112        61    0.0081            1.0000    1.3724
     arwhead   1000              5         0.0000      0.0403         3    0.0401            1.0000    1.0068
     brownal    100              5         0.0000      0.0004         1    0.0003            1.0000    1.5009
      brybnd  10000              5         0.0000     24.3815         1   23.0197            1.0592    1.0592
    clplatea    961              5         0.0000      2.9448        97    1.0576            2.7843    2.7843
    clplatec    100              5         0.0000      0.4126        87    0.0129           32.0606   32.0606
      cosine    100              5         0.0000      0.0

,name,nvar,predicted_mem,predicted_time,actual_time,best_mem,best_time,performance_ratio,raw_ratio
0,arglinb,100,87,0.0,0.000289,1,0.000229,1.000000,1.262500
1,argtrig,100,87,0.0,0.011684,61,0.008126,1.000000,1.437872
2,arwhead,1000,87,0.0,0.045967,3,0.040062,1.000000,1.147394
3,brownal,100,87,0.0,0.000553,1,0.000280,1.000000,1.976150
4,brybnd,10000,87,0.0,24.393811,1,23.019709,1.059692,1.059692
5,clplatea,961,87,0.0,1.371946,97,1.057629,1.297190,1.297190
6,clplatec,100,87,0.0,0.012869,87,0.012869,1.000000,1.000000
7,cosine,100,87,0.0,0.001197,1,0.000634,1.000000,1.887928
8,cragglvy,1000,87,0.0,0.339179,12,0.247496,1.000000,1.370442
9,cragglvy,10000,87,0.0,37.100730,6,24.499405,1.514352,1.514352
